In [217]:
from openpyxl import load_workbook
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

In [218]:
with open(f"retail-sales_total.xlsx", "wb") as f:
    x = (f'https://www.ons.gov.uk/file?'
         f'uri=%2fbusinessindustryandtrade'
         f'%2fretailindustry%2fdatasets'
         f'%2fpoundsdatatotalretailsales'
         f'%2fcurrent/poundsdata1.xlsx')
    f.write(requests.get(x).content)

with open(f"retail-sales_internet.xlsx", "wb") as f:
    x = (f'https://www.ons.gov.uk/file?'
         f'uri=%2fbusinessindustryandtrade'
         f'%2fretailindustry%2fdatasets'
         f'%2fretailsalesindexinternetsales'
         f'%2fcurrent/internetsalesreferencetables.xlsx')
    f.write(requests.get(x).content)

In [219]:
# SCRATCHPAD

# for i, row in df.iterrows():
# 	print(f"Index: {i}")
# 	print(f"{row}\n")
 
# for row in df.itertuples(index=False):
#     print(row)
    
# flat_list = [item for sublist in x for item in sublist]
# df.where(df.notnull(), None)

In [220]:
wb = load_workbook('retail-sales_internet.xlsx')
x = []

for row in wb.worksheets[2].iter_rows(min_row=7, max_col=None, max_row=None, values_only=True):
    x.append(row)

df = pd.DataFrame(x[7:],columns=x[0]).fillna(value=np.nan)
df = df.rename(columns={df.columns[0]: "date"})
df.iloc[:,0] = df.iloc[:,0].ffill()
x = list(df.columns)
x[0] = 'year'
df.columns = x
df['year'] = df['year'].astype(str).replace('\.0', '', regex=True)
df['Date'] = df[['year','date']].agg(' '.join, axis=1)
df = df.set_index('Date').drop(['year', 'date'], axis=1)
df = df * (52/12)
df.index = pd.to_datetime(df.index)
retail_sales_internet = df.round(1)
retail_sales_internet['Automotive fuel'] = 0

x = list(retail_sales_internet.columns)
retail_sales_internet.columns = [i.strip().lower() for i in x]

/home/eleanor/.miniforge3/envs/ons_data/lib/python3.10/site-packages/openpyxl/reader/drawings.py:59: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


In [221]:
wb = load_workbook('retail-sales_total.xlsx')
x = []

for row in wb.worksheets[9].iter_rows(min_row=7, max_col=None, max_row=None, values_only=True):
    x.append(row)

df = pd.DataFrame(x[4:],columns=x[0]).rename({None: 'Date'}, axis=1).set_index('Date').fillna(value=np.nan)
df = df.dropna(axis=0, how='all').drop(df.columns[12:], axis=1)
df = df.drop(df.columns[1::-1], axis=1)
df.drop('Automotive fuel', axis=1).round(1)
df.index = pd.to_datetime(df.index)
df = df.drop(df[df.index < '2008-01-01'].index)
retail_sales_total = df / 1000
x = list(retail_sales_total.columns)
retail_sales_total.columns = [i.strip().lower() for i in x]

In [222]:
retail_sales_internet.to_csv('retail_sales_internet.csv')
retail_sales_total.to_csv('retail_sales_total.csv')
retail_sales_offline = pd.DataFrame((retail_sales_total - retail_sales_internet))
retail_sales_offline.to_csv('retail_sales_offline.csv')

In [223]:
columns_to_drop = [
 'non-specialised stores',
 'textile, clothing and footwear stores',
 'household goods stores',
 'other stores',
 'automotive fuel']
rename_column = {'total':'predominantly non food','all retailing excluding automotive fuel':'total excl. fuel'}
aggregate_df_total = retail_sales_total.drop(columns_to_drop, axis=1).rename(columns=rename_column).pct_change(periods=12)
aggregate_df_offline = retail_sales_offline.drop(columns_to_drop, axis=1).rename(columns=rename_column).pct_change(periods=12)
aggregate_df_online = retail_sales_internet.drop(columns_to_drop, axis=1).rename(columns=rename_column).pct_change(periods=12)

aggregate_df_offline.drop(aggregate_df_offline[aggregate_df_offline.index < '2009-01-01'].index).to_csv('aggregate_df_offline.csv')
aggregate_df_online.drop(aggregate_df_online[aggregate_df_online.index < '2009-01-01'].index).to_csv('aggregate_df_online.csv')
aggregate_df_total.drop(aggregate_df_total[aggregate_df_total.index < '2009-01-01'].index).to_csv('aggregate_df_total.csv')

# What format does Flourish require?

## The charts
**Online and offline**
 - Total (YoY)	Food	Non-food	Non-store
 - t.csv, r.csv

**Online vs offline**
 - Online YoY%	Offline YoY%	Online Sales	Offline Sales

**YoY % by Category**
 - Total (excl. Fuel)	Food	Non-food	Non-store
 - aggregate_df_total.csv

In [225]:
r = retail_sales_internet.drop(columns_to_drop, axis=1).rename(columns=rename_column)
r['total excl. fuel'] = r['total excl. fuel'].pct_change(periods=12)
r.drop(r[r.index < '2009-01-01'].index).to_csv('r.csv')

t = retail_sales_offline.drop(columns_to_drop, axis=1).rename(columns=rename_column)
t['total excl. fuel'] = t['total excl. fuel'].pct_change(periods=12)
t.drop(t[t.index < '2009-01-01'].index).to_csv('t.csv')

q = retail_sales_internet.drop(columns_to_drop, axis=1).rename(columns=rename_column)
q = q.iloc[:, :1].rename(columns={'total excl. fuel':'total_online'})
q['total_offline'] = retail_sales_offline.drop(columns_to_drop, axis=1).rename(columns=rename_column).iloc[:, :1]
q['yoy_online'] = q.total_online.pct_change(periods=12)
q['yoy_offline'] = q.total_offline.pct_change(periods=12)
q.drop(q[q.index < '2009-01-01'].index).to_csv('q.csv')

,total_online,total_offline,yoy_online,yoy_offline
Date,,,,
2008-01-01,966.3,25599.399,NaN,NaN
2008-02-01,1022.7,20577.555,NaN,NaN
2008-03-01,1046.5,25677.095,NaN,NaN
2008-04-01,1098.9,20166.182,NaN,NaN
2008-05-01,1113.2,20993.978,NaN,NaN
...,...,...,...,...
2021-06-01,10074.1,32013.837,-0.071606,0.155805
2021-07-01,9736.1,22928.059,-0.059369,0.068475
2021-08-01,9860.9,22698.897,-0.027112,0.039230
